# Libraries

In [ ]:
# =========================================================
# Core Libraries
# =========================================================
import os
import gc
import math
import time
import copy
import random
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# Reproducibility
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    import torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# =========================================================
# Warnings
# =========================================================
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# =========================================================
# Scikit-learn
# =========================================================
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    mean_absolute_error,
    mean_squared_log_error,
    f1_score,
    precision_score
)
from sklearn.ensemble import RandomForestRegressor

# =========================================================
# Gradient Boosting
# =========================================================
import xgboost as xgb
import lightgbm as lgb

# =========================================================
# Deep Learning
# =========================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# =========================================================
# Time Series Models
# =========================================================
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet

# =========================================================
# Kaggle Input (Optional)
# =========================================================
# Uncomment if running on Kaggle
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# Main Functions

In [ ]:
# =========================================================
# Utilities
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype

        if pd.api.types.is_object_dtype(col_type) or pd.api.types.is_datetime64_any_dtype(col_type):
            continue

        if pd.api.types.is_integer_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()

            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)

        elif pd.api.types.is_float_dtype(col_type):
            df[col] = df[col].astype(np.float32)

    return df


def features_engineering(df):
    df.loc[:, 'hour'] = df['timestamp'].dt.hour
    df.loc[:, 'day'] = df['timestamp'].dt.day
    df.loc[:, 'month'] = df['timestamp'].dt.month
    df.loc[:, 'weekday'] = df['timestamp'].dt.weekday
    df['square_feet'] = np.log1p(df['square_feet'].astype(np.float32))
    df.drop(
        ['sea_level_pressure', 'wind_direction', 'wind_speed', 'year_built', 'floor_count'],
        axis=1,
        inplace=True
    )
    return df


def normalize_train_features(train_features, feature_cols, cat_threshold=5):
    binary_cols = []
    categorical_cols = []
    numeric_cols = []

    for col in feature_cols:
        unique_vals = train_features[col].dropna().unique()

        if set(unique_vals).issubset({0, 1}):
            binary_cols.append(col)
        elif np.issubdtype(train_features[col].dtype, np.integer) and len(unique_vals) <= cat_threshold:
            categorical_cols.append(col)
        else:
            numeric_cols.append(col)

    print("Numeric:", numeric_cols)
    print("Binary:", binary_cols)
    print("Categorical:", categorical_cols)

    scaler = StandardScaler()
    train_features[numeric_cols] = scaler.fit_transform(train_features[numeric_cols])

    return train_features, scaler, numeric_cols


# =========================================================
# Data Splitting
# =========================================================
def split_each_building_by_month(
    df,
    building_col,
    time_col,
    train_months=None,
    valid_months=None,
    test_months=None
):
    train_parts, valid_parts, test_parts = [], [], []

    for bld, g in df.groupby(building_col, sort=False):
        g = g.sort_values(time_col).reset_index(drop=True)
        months = g[time_col].dt.month

        if train_months is not None:
            train_parts.append(g[months.isin(train_months)])

        if valid_months is not None:
            valid_parts.append(g[months.isin(valid_months)])

        if test_months is not None:
            test_parts.append(g[months.isin(test_months)])

    train_df = pd.concat(train_parts, axis=0).reset_index(drop=True) if train_parts else pd.DataFrame()
    valid_df = pd.concat(valid_parts, axis=0).reset_index(drop=True) if valid_parts else pd.DataFrame()
    test_df = pd.concat(test_parts, axis=0).reset_index(drop=True) if test_parts else pd.DataFrame()

    return train_df, valid_df, test_df


def split_each_building(df, building_col, time_col, train_ratio=0.1, valid_ratio=0.10):
    train_parts, valid_parts, test_parts = [], [], []

    for bld, g in df.groupby(building_col, sort=False):
        g = g.sort_values(time_col).reset_index(drop=True)

        n = len(g)
        n_train = int(n * train_ratio)
        n_valid = int(n * valid_ratio)

        train_parts.append(g.iloc[:n_train])
        valid_parts.append(g.iloc[n_train:n_train + n_valid])
        test_parts.append(g.iloc[n_train + n_valid:])

    train_df = pd.concat(train_parts, axis=0).reset_index(drop=True)
    valid_df = pd.concat(valid_parts, axis=0).reset_index(drop=True)
    test_df = pd.concat(test_parts, axis=0).reset_index(drop=True)

    return train_df, valid_df, test_df


# =========================================================
# Sequence Creation
# =========================================================
def make_sequences_by_building(
    df_part,
    feature_cols,
    target_col,
    building_col,
    building_idx_col,
    seq_len=48,
    horizon=0
):
    X_seq, y_seq, b_seq = [], [], []

    for bld, g in df_part.groupby(building_col, sort=False):
        X_b = g[feature_cols].values.astype(np.float32)
        y_b = g[target_col].values.astype(np.float32)
        b_idx = int(g[building_idx_col].iloc[0])

        max_start = len(X_b) - seq_len - horizon + 1
        if max_start <= 0:
            continue

        for s in range(max_start):
            X_seq.append(X_b[s:s + seq_len])

            # predict only final target of the window
            target_idx = s + seq_len - 1 + horizon
            y_seq.append(y_b[target_idx])

            b_seq.append(b_idx)

    X_seq = np.asarray(X_seq, dtype=np.float32)
    y_seq = np.asarray(y_seq, dtype=np.float32).reshape(-1, 1)
    b_seq = np.asarray(b_seq, dtype=np.int64)

    return X_seq, y_seq, b_seq


class PrebuiltSequenceDataset(Dataset):
    def __init__(self, X, y, building_ids):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)
        building_ids = np.asarray(building_ids, dtype=np.int64)

        if y.ndim == 1:
            y = y[:, None]

        self.X = X
        self.y = y
        self.building_ids = building_ids

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # ✅ Just index (fast)
        return self.X[idx], self.y[idx], self.building_ids[idx]


# =========================================================
# Probability / Loss Functions
# =========================================================
LOG_2PI = math.log(2.0 * math.pi)

def gaussian_nll(y_true, mu, logvar):
    var = torch.exp(logvar).clamp_min(1e-6)
    return 0.5 * ((y_true - mu) ** 2 * torch.exp(-logvar) + logvar + LOG_2PI)


def standard_normal_nll(z):
    return 0.5 * (z ** 2 + LOG_2PI)

# =========================================================
# EM Step Functions
# =========================================================
@torch.no_grad()
def e_step_sample_and_weight(model, obs, y, building_ids, n_particles=5):
    particle_x = []
    log_weights = []

    for _ in range(n_particles):
        out = model.forward_sample(obs, building_ids)

        x_seq = out["x_seq"]          # [B, T, D]
        x_mu = out["x_mu"]            # [B, T, D]
        x_logvar = out["x_logvar"]    # [B, T, D]
        y_mu = out["y_mu"]            # [B, 1]
        y_logvar = out["y_logvar"]    # [B, 1]

        # use time t
        log_py = -gaussian_nll(y, y_mu, y_logvar).sum(dim=1)   # [B]

        # use time t
        log_px = -gaussian_nll(
            x_seq[:, -1, :],
            x_mu[:, -1, :],
            x_logvar[:, -1, :]
        ).sum(dim=1)

        lw = log_py

        particle_x.append(x_seq)
        log_weights.append(lw)

    particle_x = torch.stack(particle_x, dim=0)
    log_weights = torch.stack(log_weights, dim=0)

    weights = torch.softmax(log_weights, dim=0)

    N, B = weights.shape
    resampled_idx = []
    for b in range(B):
        idx_b = torch.multinomial(weights[:, b], num_samples=N, replacement=True)
        resampled_idx.append(idx_b)
    resampled_idx = torch.stack(resampled_idx, dim=1)

    resampled_x = []
    batch_idx = torch.arange(B, device=obs.device)

    for i in range(N):
        idx_i = resampled_idx[i]
        chosen_x = particle_x[idx_i, batch_idx]
        resampled_x.append(chosen_x)

    resampled_x = torch.stack(resampled_x, dim=0)
    return resampled_x


def m_step_loss(model, obs, y, building_ids, resampled_x, lambda_L2=.001):
    N = resampled_x.shape[0]

    total_y_nll = 0.0
    total_x_nll = 0.0

    x0, x_mu, x_logvar = model.encode(obs, building_ids)


    for i in range(N):
        x_seq_i = resampled_x[i]
        y_mu_i, y_logvar_i = model.decode_from_x(obs, x0, x_seq_i)

        # time t loss only
        y_nll_i = gaussian_nll(y, y_mu_i, y_logvar_i).mean()

        # final latent-state consistency only
        x_nll_i = gaussian_nll(
            x_seq_i[:, -1, :],
            x_mu[:, -1, :],
            x_logvar[:, -1, :]
        ).mean()

        total_y_nll += y_nll_i
        total_x_nll += x_nll_i

    total_y_nll /= N
    total_x_nll /= N
    mu_penalty = x_mu.pow(2).mean()
    loss = total_y_nll + total_x_nll + lambda_L2 * mu_penalty # + .01 * kl_x

    return loss, total_y_nll.detach(), total_x_nll.detach()


def train_one_epoch_em(model, loader, optimizer):
    model.train()
    total_loss = 0.0

    for obs, y, building_ids in loader:
        obs = obs.to(device)
        y = y.to(device)  # [B, 1]
        building_ids = building_ids.to(device)

        # E-step
        model.eval()
        with torch.no_grad():
            resampled_x = e_step_sample_and_weight(
                model, obs, y, building_ids,
                n_particles=N_PARTICLES
            )

        # M-step
        model.train()
        optimizer.zero_grad()

        loss, y_loss, x_loss = m_step_loss(
            model, obs, y, building_ids, resampled_x
        )

        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += float(loss.item())

    return total_loss / len(loader)


@torch.no_grad()
def valid_one_epoch_em(model, loader):
    model.eval()
    total_loss = 0.0
    total_y = 0.0
    total_x = 0.0

    for obs, y, building_ids in loader:
        obs = obs.to(device)
        y = y.to(device)
        building_ids = building_ids.to(device)

        # use same end-of-window objective as training
        resampled_x = e_step_sample_and_weight(
            model, obs, y, building_ids,
            n_particles=N_PARTICLES
        )

        loss, y_loss, x_loss, kl_loss = m_step_loss(
            model, obs, y, building_ids, resampled_x
        )

        total_loss += float(loss.item())
        total_y += float(y_loss.item())
        total_x += float(x_loss.item())

    return total_loss / len(loader)


# =========================================================
# Evaluation
# =========================================================
def Model_Evaluate(model, X_valid, y_valid, b_valid, model_name, benchmark):
    valid_preds = model.predict(X_valid, b_valid)

    y_valid = np.asarray(y_valid).reshape(-1)
    valid_preds = np.asarray(valid_preds).reshape(-1)

    y_valid_expm1 = np.expm1(y_valid)
    valid_preds_expm1 = np.expm1(valid_preds)

    valid_rmse = np.sqrt(mean_squared_error(y_valid_expm1, valid_preds_expm1))
    valid_log_rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))

    valid_r2 = r2_score(y_valid_expm1, valid_preds_expm1)
    valid_mad = np.mean(np.abs(y_valid_expm1 - valid_preds_expm1))

    denom = np.where(np.abs(y_valid_expm1) < 1e-8, 1e-8, np.abs(y_valid_expm1))
    valid_mape = np.median(np.abs((y_valid_expm1 - valid_preds_expm1) / denom)) * 100

    valid_wape = (
        np.sum(np.abs(y_valid_expm1 - valid_preds_expm1))
        / max(np.sum(np.abs(y_valid_expm1)), 1e-8)
        * 100
    )

    print(f"\nModel: {model_name}")
    print(f"Validation Log RMSE: {valid_log_rmse:.4f}")
    print(f"Validation RMSE: {valid_rmse:.4f}")
    print(f"Validation R²: {valid_r2:.4f}")
    print(f"Validation MAD: {valid_mad:.4f}")
    print(f"Validation MAPE: {valid_mape:.4f}")
    print(f"Validation WAPE: {valid_wape:.4f}%")

    return (
        y_valid_expm1.tolist(),
        valid_preds_expm1,
        valid_log_rmse,
        valid_rmse,
        valid_r2,
        valid_mad,
        valid_mape,
        valid_wape)



def compute_rmse(model, X_valid, y_valid, b_valid):
    preds = model.predict(X_valid, b_valid)

    y_valid = np.asarray(y_valid).reshape(-1)
    preds = np.asarray(preds).reshape(-1)

    logrmse = np.sqrt(np.mean((y_valid - preds) ** 2))

    y_valid = np.expm1(y_valid)
    preds = np.expm1(preds)

    rmse = np.sqrt(np.mean((y_valid - preds) ** 2))

    return rmse, logrmse


@torch.no_grad()
def mc_predict_batch(model, obs, building_ids, mc_samples=30):
    """
    Returns:
        pred_mean: [B, 1]
        pred_var : [B, 1]
    """
    model.eval()

    mus = []
    vars_ = []

    for _ in range(mc_samples):
        out = model.forward_sample(obs, building_ids)
        mus.append(out["y_mu"])                   # [B, 1]
        vars_.append(torch.exp(out["y_logvar"])) # [B, 1]

    mus = torch.stack(mus, dim=0)      # [S, B, 1]
    vars_ = torch.stack(vars_, dim=0)  # [S, B, 1]

    pred_mean = mus.mean(dim=0)
    epistemic_var = mus.var(dim=0, unbiased=False)
    aleatoric_var = vars_.mean(dim=0)
    pred_var = (epistemic_var + aleatoric_var).clamp_min(1e-8)

    return pred_mean, pred_var

# Dataset Loading and Filtering for Manufacturing

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# # Define data path (make sure your dataset is in this path)
# DATA_PATH = '/content/drive/MyDrive/SEG1PublicData'

###### The dataset in available online in Kaggle
# Original Data Source: https://www.kaggle.com/competitions/ashrae-energy-prediction/data
# Main files needed are (building_metadata.csv,sample_submission.csv, test.csv, train.csv, weather_test.csv, weather_train.csv)
# Load the Dataset from your saved folder
# some data preperation/cleaning steps are from https://www.kaggle.com/code/rebinos/energy-consumption-prediction-xgb-lgbm-and-rf/notebook


# Load the dataset
train_df = pd.read_csv(DATA_PATH+ '/train.csv')
# Remove outliers
train_df = train_df [ train_df['building_id'] != 1099 ]
train_df = train_df.query('not (building_id <= 104 & meter == 0 & timestamp <= "2016-05-20")')

## PReprocessing the data
building_df = pd.read_csv(DATA_PATH + '/building_metadata.csv')
weather_train_df = pd.read_csv(DATA_PATH + '/weather_train.csv')

## Convert timestamp to datetime
train_df['timestamp'] = pd.to_datetime(train_df['timestamp'])
weather_train_df['timestamp'] = pd.to_datetime(weather_train_df['timestamp'])


train_df = reduce_mem_usage(train_df)
weather_train_df = reduce_mem_usage(weather_train_df)
building_df = reduce_mem_usage(building_df)


# Merging building metadata with train and test data
train_df = train_df.merge(building_df, on='building_id', how='left')
 # only electricity meters
train_df=train_df[train_df.meter==0]

# Merge with weather data only if 'site_id' exists
if 'site_id' in train_df.columns and 'site_id' in weather_train_df.columns:
    train_df = train_df.merge(weather_train_df, on=['site_id', 'timestamp'], how='left')

train_df = train_df[train_df['primary_use'].isin(['Manufacturing/industrial', 'Warehouse/storage'])]

# Label encode 'primary_use' only if it exists in both datasets
if 'primary_use' in train_df.columns:
    le = LabelEncoder()
    train_df['primary_use'] = le.fit_transform(train_df['primary_use'])

# Apply feature engineering
train_df = features_engineering(train_df)

In [ ]:
train_df

,building_id,meter,timestamp,meter_reading,site_id,primary_use,square_feet,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,hour,day,month,weekday
59,164,0,2016-01-01 00:00:00,23.400000,2,1,9.465680,15.6,6.0,-5.6,NaN,0,1,1,4
60,165,0,2016-01-01 00:00:00,10.560000,2,1,8.263075,15.6,6.0,-5.6,NaN,0,1,1,4
276,383,0,2016-01-01 00:00:00,19.410000,3,1,10.215338,10.0,8.0,2.2,NaN,0,1,1,4
305,413,0,2016-01-01 00:00:00,7.020000,3,1,9.605082,10.0,8.0,2.2,NaN,0,1,1,4
317,425,0,2016-01-01 00:00:00,117.540001,3,1,12.220123,10.0,8.0,2.2,NaN,0,1,1,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11705763,1182,0,2016-12-31 23:00:00,10.000000,13,1,11.106250,-5.6,4.0,-11.1,0.0,23,31,12,5
11705778,1199,0,2016-12-31 23:00:00,21.863001,13,1,9.264829,-5.6,4.0,-11.1,0.0,23,31,12,5
11705791,1213,0,2016-12-31 23:00:00,25.601000,13,1,10.583170,-5.6,4.0,-11.1,0.0,23,31,12,5
11705917,1340,0,2016-12-31 23:00:00,17.250000,15,0,11.621950,1.7,NaN,-5.6,-1.0,23,31,12,5


# Missing Imputation

In [ ]:

# Keep building_id in both train_features and test_features
train_features = train_df.drop(columns=['meter' ]).copy()

# numeric columns to fill, excluding building_id itself
numeric_cols = train_features.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'building_id']
numeric_cols = [c for c in numeric_cols if c != 'timestamp']
numeric_cols = [c for c in numeric_cols if c != 'meter_reading']

# global fallback medians from train only
global_medians = train_features[numeric_cols].median()

# per-building medians from train only
building_medians = train_features.groupby('building_id')[numeric_cols].median()

# fill train
for col in numeric_cols:
    train_features[col] = (
        train_features[col]
        .fillna(train_features['building_id'].map(building_medians[col]))
        .fillna(global_medians[col])
    )

# optional: reduce memory
for col in numeric_cols:
    train_features[col] = train_features[col].astype(np.float32)
gc.collect()

0

# New features, Feature Normalization, and Finalizing Dataset

In [ ]:
train_features['meter_reading'] = np.log1p(train_features['meter_reading'])
# Cyclical encoding
train_features['hour_sin']   = np.sin(2 * np.pi * train_features['hour'] / 24)
train_features['hour_cos']   = np.cos(2 * np.pi * train_features['hour'] / 24)
train_features['dow_sin']    = np.sin(2 * np.pi * train_features['weekday'] / 7)
train_features['dow_cos']    = np.cos(2 * np.pi * train_features['weekday'] / 7)
train_features['month_sin']  = np.sin(2 * np.pi * train_features['month'] / 12)
train_features['month_cos']  = np.cos(2 * np.pi * train_features['month'] / 12)
# Optional binary
train_features['is_weekend'] = (train_features['weekday'] >= 5).astype(int)

# ====== YOU SET THESE ======
feature_cols = ['site_id', 'primary_use', 'square_feet',
       'air_temperature', 'cloud_coverage', 'dew_temperature',
       'precip_depth_1_hr','hour_sin', 'hour_cos','dow_sin','dow_cos','month_sin','month_cos','is_weekend'  ,  'weekday']


target_col = "meter_reading"
building_col = "building_id"
time_col = "timestamp"
unique_buildings = sorted(train_features[building_col].unique())
building_to_idx = {b: i for i, b in enumerate(unique_buildings)}

train_features["building_idx"] = train_features[building_col].map(building_to_idx)
num_buildings = len(train_features.building_idx.unique())

train_features = train_features.copy()
train_features[time_col] = pd.to_datetime(train_features[time_col])
train_features = train_features.sort_values([building_col, time_col]).reset_index(drop=True)

train_featuresNormalized, scaler, numeric_cols = normalize_train_features(
    train_features,
    feature_cols
)


Numeric: ['site_id', 'square_feet', 'air_temperature', 'cloud_coverage', 'dew_temperature', 'precip_depth_1_hr', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'weekday']
Binary: ['primary_use', 'is_weekend']
Categorical: []


# Main Model Strcuture

In [ ]:
# =========================================================
# Model
# =========================================================
class LatentStateEnergyModel(nn.Module):
    def __init__(
        self,
        obs_dim,
        num_buildings,
        latent_dim=16,
        enc_hidden=64,
        dec_hidden=64,
        enc_out_hidden=64,
        dec_out_hidden=64,
        enc_layers=2,
        dec_layers=2,
        dropout=0.1,
        y_dim=1
    ):
        super().__init__()

        self.x0_net = nn.Embedding(num_buildings, latent_dim)
        self.latent_dim = latent_dim

        self.encoder = nn.LSTM(
            input_size=obs_dim + latent_dim,
            hidden_size=enc_hidden,
            num_layers=enc_layers,
            batch_first=True,
            dropout=dropout if enc_layers > 1 else 0.0
        )

        self.enc_mu = nn.Sequential(
            nn.Linear(enc_hidden,enc_out_hidden),
            nn.ReLU(),
            nn.Linear(enc_out_hidden, latent_dim),
        )

        # self.enc_logvar = nn.Linear(enc_hidden, latent_dim)

        self.enc_logvar = nn.Sequential(
            nn.Linear(enc_hidden , enc_out_hidden),
            nn.ReLU(),
            nn.Linear(enc_out_hidden, latent_dim)
        )


        # decoder LSTM takes ONLY observations
        self.decoder = nn.LSTM(
            input_size=obs_dim,
            hidden_size=dec_hidden,
            num_layers=dec_layers,
            batch_first=True,
            dropout=dropout if dec_layers > 1 else 0.0
        )

        # y depends on decoder hidden state + x_t + x0
        self.dec_mu = nn.Sequential(
            nn.Linear(dec_hidden +obs_dim +2*latent_dim, dec_out_hidden),
            nn.ReLU(),
            nn.Linear(dec_out_hidden, y_dim)
        )

        self.dec_logvar = nn.Sequential(
            nn.Linear(dec_hidden+obs_dim +2*latent_dim, dec_out_hidden),
            nn.ReLU(),
            nn.Linear(dec_out_hidden, y_dim)
        )

    def sample_x(self, mu, logvar):
        std = torch.exp(0.5 * logvar).clamp_min(1e-6)
        eps = torch.randn_like(std)
        return mu + eps * std

    def encode(self, obs, building_ids):
        B, T, _ = obs.shape

        x0 = (self.x0_net(building_ids))
        # x0=torch.sigmoid(self.x0_net(building_ids)). # optional if you want xo to be between 0 and 1

        x0_rep = x0.unsqueeze(1).expand(-1, T, -1)

        enc_in = torch.cat([obs, x0_rep], dim=-1)

        enc_h, _ = self.encoder(enc_in)


        x_mu = 1* self.enc_mu(enc_h)
        x_logvar = self.enc_logvar(enc_h).clamp(-6.0, 1) # optional clamp to to prevent extreme values and stabilize training

        return x0, x_mu, x_logvar

    def decode_from_x(self, obs, x0, x_seq):
        B, T, _ = obs.shape
        x_last = x_seq[:, -1, :]                       # [B, latent_dim]
        x_last_rep = x_last.unsqueeze(1).expand(-1, T, -1)   # [B, T, latent_dim]
        x0_rep = x0.unsqueeze(1).expand(-1, T, -1)

        # decoder sees only observations
        dec_in = torch.cat([obs], dim=-1)
        dec_h, _ = self.decoder(dec_in)

        # # final timestep prediction
        dec_last = dec_h[:, -1, :]      # [B, dec_hidden]
        x_last = x_seq[:, -1, :]        # [B, latent_dim]
        o_last = obs[:, -1, :]#.unsqueeze(1).expand(-1, T, -1)

        head_in = torch.cat([dec_last, o_last, x_last, x0], dim=-1)

        y_mu = self.dec_mu(head_in)                  # [B, 1]
        y_logvar = self.dec_logvar(head_in).clamp(-6.0, 1) # optional clamp to prevent extreme values and stabilize training

        return y_mu, y_logvar

    def forward_sample(self, obs, building_ids):
        x0, x_mu, x_logvar = self.encode(obs, building_ids)
        x_seq = self.sample_x(x_mu, x_logvar)
        y_mu, y_logvar = self.decode_from_x(obs, x0, x_seq)

        return {
            "x0": x0,
            "x_mu": x_mu,
            "x_logvar": x_logvar,
            "x_seq": x_seq,
            "y_mu": y_mu,
            "y_logvar": y_logvar,
        }


# Training Algorithm Strcuture

In [ ]:
class EMSeq2SeqRegressor(BaseEstimator, RegressorMixin):

    def __init__(
        self,
        return_sequence=False,
        latent_dim=16,
        enc_hidden=64,
        dec_hidden=64,
        enc_out_hidden=64,
        dec_out_hidden=64,
        enc_layers=2,
        dec_layers=2,
        dropout=0.1,
        lr=1e-3,
        weight_decay=1e-5,
        batch_size=64,
        epochs=100,
        patience=10,
        loss_threshold=1e-4,
        num_buildings=None
    ):
        self.return_sequence = return_sequence
        self.latent_dim = latent_dim
        self.enc_hidden = enc_hidden
        self.dec_hidden = dec_hidden
        self.enc_out_hidden = enc_out_hidden
        self.dec_out_hidden = dec_out_hidden
        self.enc_layers = enc_layers
        self.dec_layers = dec_layers
        self.dropout = dropout
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.epochs = epochs
        self.patience = patience
        self.loss_threshold = loss_threshold
        self.num_buildings = num_buildings

        self.model_ = None
        self.history_ = None


    def fit(self, X_train, y_train, b_train, X_valid=None, y_valid=None, b_valid=None):
        X_train = np.asarray(X_train, dtype=np.float32)
        y_train = np.asarray(y_train, dtype=np.float32)
        if y_train.ndim == 1:
            y_train = y_train[:, None]

        if X_valid is None:
            X_valid = X_train
            y_valid = y_train
            b_valid = b_train
        else:
            X_valid = np.asarray(X_valid, dtype=np.float32)
            y_valid = np.asarray(y_valid, dtype=np.float32)
            if y_valid.ndim == 1:
                y_valid = y_valid[:, None]

        train_loader = DataLoader(
            PrebuiltSequenceDataset(X_train, y_train, b_train),
            batch_size=self.batch_size,num_workers=0,   # or 4
            shuffle=True
        )
        valid_loader = DataLoader(
            PrebuiltSequenceDataset(X_valid, y_valid, b_valid),
            batch_size=self.batch_size,num_workers=0,   # or 4
            shuffle=False
        )

        self.model_ = LatentStateEnergyModel(
            obs_dim=X_train.shape[-1],
            num_buildings=self.num_buildings,
            latent_dim=self.latent_dim,
            enc_hidden=self.enc_hidden,
            dec_hidden=self.dec_hidden,
            enc_out_hidden=self.enc_out_hidden,
            dec_out_hidden=self.dec_out_hidden,
            enc_layers=self.enc_layers,
            dec_layers=self.dec_layers,
            dropout=self.dropout,
            y_dim=1
        ).to(device)


        optimizer = torch.optim.Adam(
            self.model_.parameters(),
            lr=self.lr,
            weight_decay=self.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=schedulerstep,
            gamma=0.75
        )

        best_val = float("inf")
        best_state = None
        patience_counter = 0
        self.history_ = {"train_loss": [], "valid_loss": []}

        for epoch in range(1, self.epochs + 1):
            train_loss = train_one_epoch_em(self.model_, train_loader, optimizer)
            valid_loss = valid_one_epoch_em(self.model_, valid_loader)
            val_rmse,val_rmselog = compute_rmse(self, X_valid, y_valid, b_valid)

            self.history_["train_loss"].append(train_loss)
            self.history_["valid_loss"].append(valid_loss)

            print(f"Epoch {epoch:03d} | "
                f"train loss={train_loss:.6f} | "
                f"valid loss={valid_loss:.6f} | "
                f"val_LogRMSE={val_rmselog:.6f} | ")
            scheduler.step()
            if train_loss < best_val - self.loss_threshold:
                best_val = train_loss # We used training-loss-based early stopping to detect convergence of the optimization objective.
                # This can change to valid_loss (specially for grid search)
                best_state = copy.deepcopy(self.model_.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                print("Early stopping",patience_counter)
                if patience_counter >= self.patience:
                    print("Early stopping")
                    break

        if best_state is not None:
            self.model_.load_state_dict(best_state)

        return self

    def predict(self, X, building_ids):
      self.model_.eval()

      X = np.asarray(X, dtype=np.float32)
      building_ids = np.asarray(building_ids, dtype=np.int64)
      dummy_y = np.zeros((X.shape[0], 1), dtype=np.float32)

      loader = DataLoader(
          PrebuiltSequenceDataset(X, dummy_y, building_ids),
          batch_size=self.batch_size,
          shuffle=False
      )

      all_preds = []

      with torch.no_grad():
          for obs, _, b_ids in loader:
              obs = obs.to(device)
              b_ids = b_ids.to(device)

              x0, x_mu, x_logvar = self.model_.encode(obs, b_ids)
              # print(x_mu)
              y_mu, _ = self.model_.decode_from_x(obs, x0, x_mu)

              all_preds.append(y_mu.cpu().numpy())

      pred_mean = np.concatenate(all_preds, axis=0).reshape(-1)
      return pred_mean

    def predict_dist(self, X, building_ids):
        self.model_.eval()

        X = np.asarray(X, dtype=np.float32)
        building_ids = np.asarray(building_ids, dtype=np.int64)
        dummy_y = np.zeros((X.shape[0], 1), dtype=np.float32)

        loader = DataLoader(
            PrebuiltSequenceDataset(X, dummy_y, building_ids),
            batch_size=self.batch_size,num_workers=0,   # or 4
            shuffle=False
        )

        all_mean = []
        all_std = []

        with torch.no_grad():
            for obs, _, b_ids in loader:
                obs = obs.to(device)
                b_ids = b_ids.to(device)

                mu, var = mc_predict_batch(self.model_, obs, b_ids, mc_samples=MC_SAMPLES)
                std = torch.sqrt(var)

                all_mean.append(mu.cpu().numpy())   # [B, 1]
                all_std.append(std.cpu().numpy())   # [B, 1]

        pred_mean = np.concatenate(all_mean, axis=0).reshape(-1)
        pred_std = np.concatenate(all_std, axis=0).reshape(-1)
        return pred_mean, pred_std

    def predict_latent(self, X, building_ids, return_full_seq=False):
      self.model_.eval()

      X = np.asarray(X, dtype=np.float32)
      building_ids = np.asarray(building_ids, dtype=np.int64)
      dummy_y = np.zeros((X.shape[0], 1), dtype=np.float32)

      loader = DataLoader(
          PrebuiltSequenceDataset(X, dummy_y, building_ids),
          batch_size=self.batch_size,
          shuffle=False
      )

      all_x_last = []
      all_x_seq = []

      with torch.no_grad():
          for obs, _, b_ids in loader:
              obs = obs.to(device)
              b_ids = b_ids.to(device)

              # EXACT SAME as predict()
              x0, x_mu, x_logvar = self.model_.encode(obs, b_ids)

              # latent used by decoder
              x_last = x_mu[:, -1, :]   # [B, latent_dim]
              # print(x_last)

              all_x_last.append(x_last.cpu().numpy())

              if return_full_seq:
                  all_x_seq.append(x_mu.cpu().numpy())  # [B, T, latent_dim]

      x_last = np.concatenate(all_x_last, axis=0)

      if return_full_seq:
          x_seq = np.concatenate(all_x_seq, axis=0)
          return x_last, x_seq

      return x_last



# Model Parameters and Hyperparameters

In [ ]:
# =========================================================
# Hyperparameters
# =========================================================
# Reproducibility / Device
SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
set_seed(SEED)
horizon = 0


# Training Parameters
EPOCHS = 100   # maximum epochs
PATIENCE = 5
WEIGHT_DECAY = 1e-5
Optimzier = 'Adam'
Loss_threshold = 0.01
Lambda_L2=0.001
schedulerstep=100 # optional, not used
# Model Architecture

# # Model Hyperparameters
LATENT_DIM = 2
ENC_HIDDEN = 128  # LSTM Encoder
DEC_HIDDEN = 64  # LSTM Decoder
Enc_Out_Hidden=64 # decoder middel layer
Dec_Out_Hidden=64 # encoder middle layer
ENC_LAYERS = 2
DEC_LAYERS = 1
DROPOUT = 0.01
N_PARTICLES = 5
MC_SAMPLES = 1
seq_len=12
LR=0.005
BATCH_SIZE=128




Device: cuda


# Train and Test

In [ ]:

if 'building_id' in feature_cols:
    feature_cols.remove('building_id')

# =========================================================
# Example split and training
# Define Train and Test Months (inclusing all buildings)
# =========================================================
train_df, valid_df, test_df = split_each_building_by_month(
    train_featuresNormalized,
    building_col="building_id",
    time_col="timestamp",
    train_months=[1,2,3,4,5,6],
    valid_months=[7,8],
    test_months=None
)

X_train_seq, y_train_seq, b_train_seq = make_sequences_by_building(
    train_df, feature_cols, target_col, building_col, "building_idx",
    seq_len=seq_len, horizon=horizon)

X_valid_seq, y_valid_seq, b_valid_seq = make_sequences_by_building(
    valid_df, feature_cols, target_col, building_col, "building_idx",
    seq_len=seq_len, horizon=horizon)

if len(test_df) > 0:
    X_test, y_test, b_test = make_sequences_by_building(
        test_df,
        feature_cols,
        target_col,
        building_col,
        building_idx_col="building_idx",
        seq_len=seq_len, horizon=horizon)
else:
    X_test, y_test, b_test = np.array([]), np.array([]), np.array([])


em_model = EMSeq2SeqRegressor(
    return_sequence=False,
    latent_dim=LATENT_DIM,
    enc_hidden=ENC_HIDDEN,
    dec_hidden=DEC_HIDDEN,
    enc_out_hidden=Enc_Out_Hidden,
    dec_out_hidden=Dec_Out_Hidden,
    enc_layers=ENC_LAYERS,
    dec_layers=DEC_LAYERS,
    dropout=DROPOUT,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    patience=PATIENCE,
    loss_threshold=Loss_threshold,
    num_buildings=num_buildings
)
em_model.fit(X_train_seq, y_train_seq, b_train_seq, X_valid_seq, y_valid_seq, b_valid_seq)

_,_, log_rmse, rmse, r2, mad, mape, wape = Model_Evaluate(
        em_model,
        X_valid_seq,
        y_valid_seq,
        b_valid_seq,
        model_name="em_model",
        benchmark="no"
    )




Epoch 001 | train loss=0.935866 | valid loss=0.384051 | val_LogRMSE=0.454343 | 
Epoch 002 | train loss=0.173804 | valid loss=0.547550 | val_LogRMSE=0.426633 | 
Epoch 003 | train loss=-0.040088 | valid loss=0.742114 | val_LogRMSE=0.409651 | 
Epoch 004 | train loss=-0.095633 | valid loss=0.464463 | val_LogRMSE=0.426487 | 
Epoch 005 | train loss=-0.146717 | valid loss=0.564633 | val_LogRMSE=0.395336 | 
Epoch 006 | train loss=-0.227573 | valid loss=1.678521 | val_LogRMSE=0.423147 | 
Epoch 007 | train loss=-0.252322 | valid loss=1.866093 | val_LogRMSE=0.466274 | 
Epoch 008 | train loss=-0.273702 | valid loss=0.792224 | val_LogRMSE=0.395279 | 
Epoch 009 | train loss=-0.314382 | valid loss=1.022120 | val_LogRMSE=0.392152 | 
Epoch 010 | train loss=-0.372038 | valid loss=1.592482 | val_LogRMSE=0.397864 | 
Epoch 011 | train loss=-0.401697 | valid loss=1.089809 | val_LogRMSE=0.375562 | 
Epoch 012 | train loss=-0.416097 | valid loss=1.254267 | val_LogRMSE=0.421390 | 
Epoch 013 | train loss=-0.4114

#  Five Fold Cross-validation


In [ ]:
import numpy as np
import pandas as pd

if 'building_id' in feature_cols:
    feature_cols.remove('building_id')

# =========================================================
# 5-fold rolling cross validation
# =========================================================
cv_folds = [
    {"fold": 1, "train_months": [1, 2, 3, 4, 5, 6],  "valid_months": [7, 8]},
    {"fold": 2, "train_months": [2, 3, 4, 5, 6, 7],  "valid_months": [8, 9]},
    {"fold": 3, "train_months": [3, 4, 5, 6, 7, 8],  "valid_months": [9, 10]},
    {"fold": 4, "train_months": [4, 5, 6, 7, 8, 9],  "valid_months": [10, 11]},
    {"fold": 5, "train_months": [5, 6, 7, 8, 9, 10], "valid_months": [11, 12]},
]

fold_results = []

for fold_cfg in cv_folds:
    fold_num = fold_cfg["fold"]
    train_months = fold_cfg["train_months"]
    valid_months = fold_cfg["valid_months"]

    print(f"\n{'='*60}")
    print(f"Fold {fold_num}")
    print(f"Train months: {train_months}")
    print(f"Valid months: {valid_months}")
    print(f"{'='*60}")

    # -----------------------------------------------------
    # Split data
    # -----------------------------------------------------
    train_df, valid_df, test_df = split_each_building_by_month(
        train_featuresNormalized,
        building_col="building_id",
        time_col="timestamp",
        train_months=train_months,
        valid_months=valid_months,
        test_months=None
    )

    # -----------------------------------------------------
    # Build sequences
    # -----------------------------------------------------
    X_train_seq, y_train_seq, b_train_seq = make_sequences_by_building(
        train_df,
        feature_cols,
        target_col,
        building_col,
        "building_idx",
        seq_len=seq_len,
        horizon=horizon
    )

    X_valid_seq, y_valid_seq, b_valid_seq = make_sequences_by_building(
        valid_df,
        feature_cols,
        target_col,
        building_col,
        "building_idx",
        seq_len=seq_len,
        horizon=horizon
    )

    # Skip fold if no sequences were generated
    if len(X_train_seq) == 0 or len(X_valid_seq) == 0:
        print(f"Skipping fold {fold_num} because train/valid sequences are empty.")
        continue

    # -----------------------------------------------------
    # Initialize a fresh model for each fold
    # -----------------------------------------------------
    em_model = EMSeq2SeqRegressor(
        return_sequence=False,
        latent_dim=LATENT_DIM,
        enc_hidden=ENC_HIDDEN,
        dec_hidden=DEC_HIDDEN,
        enc_out_hidden=Enc_Out_Hidden,
        dec_out_hidden=Dec_Out_Hidden,
        enc_layers=ENC_LAYERS,
        dec_layers=DEC_LAYERS,
        dropout=DROPOUT,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        patience=PATIENCE,
        loss_threshold=Loss_threshold,
        num_buildings=num_buildings
    )

    # -----------------------------------------------------
    # Train
    # -----------------------------------------------------
    em_model.fit(
        X_train_seq, y_train_seq, b_train_seq,
        X_valid_seq, y_valid_seq, b_valid_seq
    )

    # -----------------------------------------------------
    # Evaluate on validation fold
    # -----------------------------------------------------
    _, _, log_rmse, rmse, r2, mad, mape, wape = Model_Evaluate(
        em_model,
        X_valid_seq,
        y_valid_seq,
        b_valid_seq,
        model_name=f"em_model_fold_{fold_num}",
        benchmark="no"
    )

    fold_results.append({
        "fold": fold_num,
        "train_months": train_months,
        "valid_months": valid_months,
        "log_rmse": log_rmse,
        "rmse": rmse,
        "r2": r2,
        "mad": mad,
        "mape": mape,
        "wape": wape
    })

# =========================================================
# Aggregate fold results
# =========================================================
results_df = pd.DataFrame(fold_results)

print("\nPer-fold results:")
print(results_df)

metric_cols = ["log_rmse", "rmse", "r2", "mad", "mape", "wape"]

summary_df = pd.DataFrame({
    "metric": metric_cols,
    "mean": [results_df[col].mean() for col in metric_cols],
    "std":  [results_df[col].std(ddof=1) for col in metric_cols]
})

print("\nCross-validation summary (mean ± std):")
print(summary_df)

# Optional pretty print
for _, row in summary_df.iterrows():
    print(f"{row['metric']}: {row['mean']:.6f} ± {row['std']:.6f}")


Fold 1
Train months: [1, 2, 3, 4, 5, 6]
Valid months: [7, 8]
Epoch 001 | train loss=2.859400 | valid loss=2.114456 | val_LogRMSE=0.513705 | 
Epoch 002 | train loss=1.819449 | valid loss=2.372446 | val_LogRMSE=0.487746 | 
Epoch 003 | train loss=1.593314 | valid loss=3.491307 | val_LogRMSE=0.464810 | 
Epoch 004 | train loss=1.546088 | valid loss=3.231638 | val_LogRMSE=0.470765 | 
Epoch 005 | train loss=1.402326 | valid loss=3.407651 | val_LogRMSE=0.455871 | 
Epoch 006 | train loss=1.337643 | valid loss=3.467382 | val_LogRMSE=0.454651 | 
Epoch 007 | train loss=1.311484 | valid loss=3.017603 | val_LogRMSE=0.420103 | 
Epoch 008 | train loss=1.278317 | valid loss=4.030912 | val_LogRMSE=0.444533 | 
Epoch 009 | train loss=1.247312 | valid loss=3.745823 | val_LogRMSE=0.436970 | 
Epoch 010 | train loss=1.254085 | valid loss=3.256989 | val_LogRMSE=0.449519 | 
Early stopping 1
Epoch 011 | train loss=1.227322 | valid loss=5.718689 | val_LogRMSE=0.449875 | 
Epoch 012 | train loss=1.208703 | valid l

# Results of 5-fold (different latent states)

In [ ]:
latent 1

Per-fold results:
   fold         train_months valid_months  log_rmse       rmse        r2  \
0     1   [1, 2, 3, 4, 5, 6]       [7, 8]  0.400893  37.369016  0.774129
1     2   [2, 3, 4, 5, 6, 7]       [8, 9]  0.466955  35.711049  0.788859
2     3   [3, 4, 5, 6, 7, 8]      [9, 10]  0.437522  30.673358  0.834409
3     4   [4, 5, 6, 7, 8, 9]     [10, 11]  0.459333  36.512533  0.771170
4     5  [5, 6, 7, 8, 9, 10]     [11, 12]  0.455514  39.010112  0.768657

         mad       mape       wape
0  19.931898  17.274712  26.377396
1  18.095551  18.226210  24.840300
2  15.696186  19.422935  22.918640
3  17.971388  19.118504  26.763809
4  20.961399  23.412203  29.895002

Cross-validation summary (mean ± std):
     metric       mean       std
0  log_rmse   0.444044  0.026433
1      rmse  35.855214  3.144418
2        r2   0.787445  0.027399
3       mad  18.531284  2.025358
4      mape  19.490911  2.346851
5      wape  26.159031  2.579249
log_rmse: 0.444044 ± 0.026433
rmse: 35.855214 ± 3.144418
r2: 0.787445 ± 0.027399
mad: 18.531284 ± 2.025358
mape: 19.490911 ± 2.346851
wape: 26.159031 ± 2.579249


LATENT_DIM = 2 ENC_HIDDEN = 128  # LSTM Encoder DEC_HIDDEN = 64
# LSTM Decoder Enc_Out_Hidden=64 # decoder middel layer Dec_Out_Hidden=64 # encoder middle layer ENC_LAYERS = 2 DEC_LAYERS = 1 DROPOUT = 0.01 N_PARTICLES = 5 MC_SAMPLES = 1 seq_len=12 LR=0.005 BATCH_SIZE=128

Per-fold results:
   fold         train_months valid_months  log_rmse       rmse        r2  \
0     1   [1, 2, 3, 4, 5, 6]       [7, 8]  0.400545  34.278791  0.809941
1     2   [2, 3, 4, 5, 6, 7]       [8, 9]  0.423370  32.975011  0.819973
2     3   [3, 4, 5, 6, 7, 8]      [9, 10]  0.448437  34.429268  0.791373
3     4   [4, 5, 6, 7, 8, 9]     [10, 11]  0.400483  30.996687  0.835085
4     5  [5, 6, 7, 8, 9, 10]     [11, 12]  0.415890  34.864295  0.815216

         mad       mape       wape
0  17.928497  14.351028  23.726143
1  17.526001  17.366787  24.058464
2  17.399134  16.443289  25.405180
3  15.888006  17.924400  23.661142
4  18.325455  22.182878  26.135639

Cross-validation summary (mean ± std):
     metric       mean       std
0  log_rmse   0.417745  0.019817
1      rmse  33.508810  1.571079
2        r2   0.814318  0.015888
3       mad  17.413418  0.926941
4      mape  17.653675  2.874099
5      wape  24.597315  1.111846
log_rmse: 0.417745 ± 0.019817
rmse: 33.508810 ± 1.571079
r2: 0.814318 ± 0.015888
mad: 17.413418 ± 0.926941
mape: 17.653675 ± 2.874099
wape: 24.597315 ± 1.111846

# latenst 3

Per-fold results:
   fold         train_months valid_months  log_rmse       rmse        r2  \
0     1   [1, 2, 3, 4, 5, 6]       [7, 8]  0.399107  30.216665  0.852317
1     2   [2, 3, 4, 5, 6, 7]       [8, 9]  0.415120  31.261636  0.838195
2     3   [3, 4, 5, 6, 7, 8]      [9, 10]  0.429734  30.658744  0.834567
3     4   [4, 5, 6, 7, 8, 9]     [10, 11]  0.544790  40.800440  0.714268
4     5  [5, 6, 7, 8, 9, 10]     [11, 12]  0.446080  38.240961  0.777690

         mad       mape       wape
0  16.685493  16.008713  22.081181
1  15.678053  15.904066  21.521729
2  15.855950  18.260105  23.151918
3  23.634048  27.225336  35.196899
4  19.758722  21.605116  28.179752

Cross-validation summary (mean ± std):
     metric       mean       std
0  log_rmse   0.446966  0.057384
1      rmse  34.235689  4.922658
2        r2   0.803407  0.057403
3       mad  18.322453  3.392346
4      mape  19.800667  4.751776
5      wape  26.026297  5.763125
log_rmse: 0.446966 ± 0.057384
rmse: 34.235689 ± 4.922658
r2: 0.803407 ± 0.057403
mad: 18.322453 ± 3.392346
mape: 19.800667 ± 4.751776
wape: 26.026297 ± 5.763125


latent 4
Per-fold results:
   fold         train_months valid_months  log_rmse       rmse        r2  \
0     1   [1, 2, 3, 4, 5, 6]       [7, 8]  0.400294  35.055064  0.801235
1     2   [2, 3, 4, 5, 6, 7]       [8, 9]  0.393869  28.884397  0.861868
2     3   [3, 4, 5, 6, 7, 8]      [9, 10]  0.440774  32.742416  0.811316
3     4   [4, 5, 6, 7, 8, 9]     [10, 11]  0.414080  32.509734  0.818592
4     5  [5, 6, 7, 8, 9, 10]     [11, 12]  0.413999  34.574596  0.818274

         mad       mape       wape
0  18.682938  16.238665  24.724548
1  13.860622  14.387202  19.026890
2  16.614630  17.376997  24.259693
3  17.250593  20.352793  25.690369
4  18.930834  22.500668  26.999023

Cross-validation summary (mean ± std):
     metric       mean       std
0  log_rmse   0.412603  0.018029
1      rmse  32.753242  2.431208
2        r2   0.822257  0.023238
3       mad  17.067923  2.037421
4      mape  18.171265  3.247604
5      wape  24.140104  3.044568
log_rmse: 0.412603 ± 0.018029
rmse: 32.753242 ± 2.431208
r2: 0.822257 ± 0.023238
mad: 17.067923 ± 2.037421
mape: 18.171265 ± 3.247604
wape: 24.140104 ± 3.044568


latent  5
Per-fold results:
   fold         train_months valid_months  log_rmse       rmse        r2  \
0     1   [1, 2, 3, 4, 5, 6]       [7, 8]  0.383651  32.908201  0.824835
1     2   [2, 3, 4, 5, 6, 7]       [8, 9]  0.421631  32.193109  0.828409
2     3   [3, 4, 5, 6, 7, 8]      [9, 10]  0.455286  34.314871  0.792758
3     4   [4, 5, 6, 7, 8, 9]     [10, 11]  0.407364  34.537593  0.795255
4     5  [5, 6, 7, 8, 9, 10]     [11, 12]  0.458551  41.044658  0.743896

         mad       mape       wape
0  17.184870  13.968293  22.742043
1  16.801540  19.355871  23.063974
2  17.224897  20.384277  25.150772
3  18.233780  20.571138  27.154577
4  22.099152  25.658878  31.517658

Cross-validation summary (mean ± std):
     metric       mean       std
0  log_rmse   0.425297  0.031916
1      rmse  34.999686  3.516937
2        r2   0.797031  0.033920
3       mad  18.308849  2.184164
4      mape  19.987692  4.162103
5      wape  25.925806  3.595337
log_rmse: 0.425297 ± 0.031916
rmse: 34.999686 ± 3.516937
r2: 0.797031 ± 0.033920
mad: 18.308849 ± 2.184164
mape: 19.987692 ± 4.162103
wape: 25.925806 ± 3.595337

